# 📊 Embryos with Prescription Wide Metrics: DuckDB vs AWS Athena

This notebook connects to both the local **DuckDB** database (`huntington_data_lake.duckdb`) and **AWS Athena (Production)** (`gold_huntington_prod`) to execute data quality and validation queries on the `embryos_with_prescription_wide` table.

### Key Metrics Reconciled:
1. **Total Row Count** and **Unique Oocito IDs**
2. **Unique Prontuario Counts**
3. **Per-Year procedural breakdown** (`micro_data_procedimento` year)
4. **Per-Clinic breakdown** (`patient_unit_huntington` / `embryo_unit_huntington`)
5. **Medication Group Counts** (Side-by-side reconciliation of pivoted columns `*_dose`)
6. **Discrepancy Analysis** (Oocito IDs exclusive to local DuckDB or AWS Athena)
7. **Value Diagnostics** (Sample level dose comparison)

In [ ]:
import duckdb
import pandas as pd
import numpy as np
from pyathena import connect
import warnings
warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------
DUCKDB_PATH = '../../database/huntington_data_lake.duckdb'
ATHENA_REGION = 'sa-east-1'
ATHENA_WORKGROUP = 'datalake-admins'
ATHENA_SCHEMA = 'gold_huntington_prod'
STAGING_SCHEMA = 'gold_huntington_staging'

print("Connecting to DuckDB...")
duck_con = duckdb.connect(DUCKDB_PATH, read_only=True)

print(f"Connecting to AWS Athena ({ATHENA_SCHEMA})...")
try:
    ath_con = connect(
        region_name=ATHENA_REGION,
        work_group=ATHENA_WORKGROUP,
        schema_name=ATHENA_SCHEMA
    )
    ath_cur = ath_con.cursor()
    print("Successfully connected to AWS Athena.")
except Exception as e:
    print(f"Warning: Could not connect to Athena: {e}")
    ath_cur = None

## 🔍 Part 1: DuckDB Queries (Local Gold Layer)

Executing local validation queries on `gold.embryos_with_prescription_wide` in DuckDB.

In [ ]:
# 1. Overall Totals (DuckDB)
query_duck_overall = '''
SELECT 
    'TOTAL' as grouping_type,
    'ALL' as grouping_value,
    COUNT(*) as total_rows,
    COUNT(DISTINCT oocito_id) as unique_oocito_id,
    COUNT(*) - COUNT(DISTINCT oocito_id) as duplicate_oocito_id_count,
    ROUND((COUNT(*) - COUNT(DISTINCT oocito_id)) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_duplicate_oocito_id,
    COUNT(DISTINCT micro_prontuario) as unique_prontuario
FROM gold.embryos_with_prescription_wide
'''
df_duck_overall = duck_con.execute(query_duck_overall).df()
print("--- DuckDB Overall Totals ---")
display(df_duck_overall)

# 2. Per Year Breakdown (DuckDB)
query_duck_year = '''
SELECT 
    YEAR(CAST(micro_data_procedimento AS DATE)) as procedimento_year,
    COUNT(*) as total_rows,
    COUNT(DISTINCT oocito_id) as unique_oocito_id,
    COUNT(*) - COUNT(DISTINCT oocito_id) as duplicate_oocito_id_count,
    COUNT(DISTINCT micro_prontuario) as unique_prontuario
FROM gold.embryos_with_prescription_wide
WHERE micro_data_procedimento IS NOT NULL
GROUP BY YEAR(CAST(micro_data_procedimento AS DATE))
ORDER BY procedimento_year DESC
'''
df_duck_year = duck_con.execute(query_duck_year).df()
print("\n--- DuckDB Per Year Breakdown ---")
display(df_duck_year)

# 3. Per Unidade Breakdown (DuckDB)
query_duck_unidade = '''
SELECT 
    COALESCE(patient_unit_huntington, 'NA / Desconhecido') as patient_unit_huntington,
    COUNT(*) as total_rows,
    COUNT(DISTINCT oocito_id) as unique_oocito_id,
    COUNT(*) - COUNT(DISTINCT oocito_id) as duplicate_oocito_id_count,
    COUNT(DISTINCT micro_prontuario) as unique_prontuario
FROM gold.embryos_with_prescription_wide
GROUP BY COALESCE(patient_unit_huntington, 'NA / Desconhecido')
ORDER BY total_rows DESC
'''
df_duck_unidade = duck_con.execute(query_duck_unidade).df()
print("\n--- DuckDB Per Unidade Breakdown ---")
display(df_duck_unidade)

## ☁️ Part 2: AWS Athena Queries (Production Gold Layer)

Executing queries against AWS Athena (`{ATHENA_SCHEMA}.embryos_with_prescription_wide`).

In [ ]:
# Athena Queries Execution
if ath_cur is not None:
    # 1. Overall Totals (Athena)
    query_ath_overall = f'''
    SELECT 
        'TOTAL' as grouping_type,
        'ALL' as grouping_value,
        COUNT(*) as total_rows,
        COUNT(DISTINCT oocito_id) as unique_oocito_id,
        COUNT(*) - COUNT(DISTINCT oocito_id) as duplicate_oocito_id_count,
        ROUND(CAST(COUNT(*) - COUNT(DISTINCT oocito_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_duplicate_oocito_id,
        COUNT(DISTINCT micro_prontuario) as unique_prontuario
    FROM {ATHENA_SCHEMA}.embryos_with_prescription_wide
    '''
    try:
        ath_cur.execute(query_ath_overall)
        df_ath_overall = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
        print("--- AWS Athena Overall Totals ---")
        display(df_ath_overall)
    except Exception as e:
        print(f"Error running Athena overall query: {e}")
        df_ath_overall = pd.DataFrame()

    # 2. Per Year Breakdown (Athena)
    query_ath_year = f'''
    SELECT 
        YEAR(CAST(micro_data_procedimento AS DATE)) as procedimento_year,
        COUNT(*) as total_rows,
        COUNT(DISTINCT oocito_id) as unique_oocito_id,
        COUNT(*) - COUNT(DISTINCT oocito_id) as duplicate_oocito_id_count,
        COUNT(DISTINCT micro_prontuario) as unique_prontuario
    FROM {ATHENA_SCHEMA}.embryos_with_prescription_wide
    WHERE micro_data_procedimento IS NOT NULL
    GROUP BY YEAR(CAST(micro_data_procedimento AS DATE))
    ORDER BY procedimento_year DESC
    '''
    try:
        ath_cur.execute(query_ath_year)
        df_ath_year = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
        print("\n--- AWS Athena Per Year Breakdown ---")
        display(df_ath_year)
    except Exception as e:
        print(f"Error running Athena year query: {e}")
        df_ath_year = pd.DataFrame()

    # 3. Per Unidade Breakdown (Athena - using embryo_unit_huntington)
    query_ath_unidade = f'''
    SELECT 
        COALESCE(embryo_unit_huntington, 'NA / Desconhecido') as patient_unit_huntington,
        COUNT(*) as total_rows,
        COUNT(DISTINCT oocito_id) as unique_oocito_id,
        COUNT(*) - COUNT(DISTINCT oocito_id) as duplicate_oocito_id_count,
        COUNT(DISTINCT micro_prontuario) as unique_prontuario
    FROM {ATHENA_SCHEMA}.embryos_with_prescription_wide
    GROUP BY COALESCE(embryo_unit_huntington, 'NA / Desconhecido')
    ORDER BY total_rows DESC
    '''
    try:
        ath_cur.execute(query_ath_unidade)
        df_ath_unidade = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
        print("\n--- AWS Athena Per Unidade Breakdown ---")
        display(df_ath_unidade)
    except Exception as e:
        print(f"Error running Athena unidade query: {e}")
        df_ath_unidade = pd.DataFrame()
else:
    print("Athena cursor unavailable. Skipping Athena queries.")
    df_ath_overall = pd.DataFrame()
    df_ath_year = pd.DataFrame()
    df_ath_unidade = pd.DataFrame()

## ⚖️ Part 3: Side-by-Side Reconciliation & Delta Comparison

Compares DuckDB vs Athena totals and highlights differences.

In [ ]:
# Build Reconciliation Summary
if not df_duck_overall.empty:
    duck_summary = df_duck_overall.iloc[0].to_dict()
    
    if not df_ath_overall.empty:
        ath_summary = df_ath_overall.iloc[0].to_dict()
    else:
        ath_summary = {k: None for k in duck_summary.keys()}
        
    metrics = [
        'total_rows',
        'unique_oocito_id',
        'duplicate_oocito_id_count',
        'unique_prontuario'
    ]
    
    recon_rows = []
    for m in metrics:
        v_duck = duck_summary.get(m, 0)
        v_ath = ath_summary.get(m, 0) if ath_summary.get(m) is not None else 0
        diff = (v_duck - v_ath) if (v_duck is not None and v_ath is not None) else None
        pct_match = (100.0 * min(v_duck, v_ath) / max(v_duck, v_ath)) if (v_duck and v_ath) else (100.0 if v_duck == v_ath else 0.0)
        
        recon_rows.append({
            'Metric': m,
            'DuckDB (Local)': v_duck,
            'AWS Athena (Prod)': v_ath,
            'Delta (Duck - Ath)': diff,
            'Match %': f"{pct_match:.2f}%" if pct_match is not None else 'N/A'
        })
        
    df_recon = pd.DataFrame(recon_rows)
    print("=========================================================================")
    print("               SUMMARY RECONCILIATION: DUCKDB VS ATHENA                  ")
    print("=========================================================================")
    display(df_recon)

## 💊 Part 4: Medication Group Side-by-Side Reconciliation

Extracts all pivoted medication groups and compares their non-null counts side-by-side.

In [ ]:
print("Extracting pivoted medication groups...")
desc_df = duck_con.execute("DESCRIBE gold.embryos_with_prescription_wide").df()
dose_cols = [col for col in desc_df['column_name'].tolist() if col.endswith('_dose')]
med_groups = sorted([col.replace('_dose', '') for col in dose_cols])
print(f"Found {len(med_groups)} medication groups in local DuckDB.")

# 1. Fetch counts from DuckDB using double quotes for columns
duck_selects = [f'COUNT(CASE WHEN "{g}_dose" IS NOT NULL THEN oocito_id END) as "{g}_count"' for g in med_groups]
query_duck_meds = f"SELECT {', '.join(duck_selects)} FROM gold.embryos_with_prescription_wide"
df_duck_meds = duck_con.execute(query_duck_meds).df()
duck_med_counts = {g: df_duck_meds.iloc[0][f"{g}_count"] for g in med_groups}

# 2. Fetch counts from AWS Athena (handling lowercase mapping and missing columns in Athena)
ath_med_counts = {}
if ath_cur is not None:
    try:
        # Get actual column names in Athena
        ath_cur.execute(f"SELECT * FROM {ATHENA_SCHEMA}.embryos_with_prescription_wide LIMIT 0")
        ath_cols = {desc[0].lower() for desc in ath_cur.description}
        
        # Only query columns that exist in Athena
        ath_selects = []
        queried_groups = []
        for g in med_groups:
            g_ath = g.lower()
            ath_col_name = f"{g_ath}_dose"
            if ath_col_name in ath_cols:
                ath_selects.append(f'COUNT(CASE WHEN "{ath_col_name}" IS NOT NULL THEN oocito_id END) as "{g}_count"')
                queried_groups.append(g)
            else:
                ath_med_counts[g] = 0  # Not in Athena
                
        if ath_selects:
            query_ath_meds = f"SELECT {', '.join(ath_selects)} FROM {ATHENA_SCHEMA}.embryos_with_prescription_wide"
            ath_cur.execute(query_ath_meds)
            df_ath_meds = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
            for g in queried_groups:
                ath_med_counts[g] = df_ath_meds.iloc[0][f"{g}_count"]
    except Exception as e:
        print(f"Error fetching Athena medication counts: {e}")
        for g in med_groups:
            if g not in ath_med_counts:
                ath_med_counts[g] = 0
else:
    print("Athena cursor unavailable. Setting counts to 0.")
    ath_med_counts = {g: 0 for g in med_groups}

# 3. Merge and Reconcile
med_recon_rows = []
for g in med_groups:
    v_duck = duck_med_counts.get(g, 0)
    v_ath = ath_med_counts.get(g, 0)
    diff = v_duck - v_ath
    pct_match = (100.0 * min(v_duck, v_ath) / max(v_duck, v_ath)) if (v_duck and v_ath) else (100.0 if v_duck == v_ath else 0.0)
    
    med_recon_rows.append({
        'Medication Group': g,
        'DuckDB (Local)': v_duck,
        'AWS Athena (Prod)': v_ath,
        'Delta (Duck - Ath)': diff,
        'Match %': f"{pct_match:.2f}%"
    })

df_med_recon = pd.DataFrame(med_recon_rows).sort_values(by='DuckDB (Local)', ascending=False)
print("\n=========================================================================")
print("        MEDICATION GROUP RECONCILIATION: DUCKDB VS ATHENA                ")
print("=========================================================================")
display(df_med_recon)

## 🔎 Part 5: Oocito Discrepancy Analysis (Missing Records)

Identifies `oocito_id`s that exist exclusively in DuckDB or exclusively in AWS Athena, displaying sample records for detailed investigation.

In [ ]:
print("Fetching set of unique oocito_ids from DuckDB...")
duck_oocito_ids = set(duck_con.execute("SELECT oocito_id FROM gold.embryos_with_prescription_wide").df()["oocito_id"].tolist())
print(f"Total unique oocito_ids in DuckDB: {len(duck_oocito_ids):,}")

if ath_cur is not None:
    print("Fetching set of unique oocito_ids from Athena (gold_huntington_prod)...")
    try:
        ath_cur.execute(f"SELECT oocito_id FROM {ATHENA_SCHEMA}.embryos_with_prescription_wide")
        ath_oocito_ids = set([row[0] for row in ath_cur.fetchall()])
        print(f"Total unique oocito_ids in Athena: {len(ath_oocito_ids):,}")
    except Exception as e:
        print(f"Error fetching oocito_ids from Athena: {e}")
        ath_oocito_ids = set()
else:
    ath_oocito_ids = set()

if len(ath_oocito_ids) > 0:
    only_in_duck = duck_oocito_ids - ath_oocito_ids
    only_in_ath = ath_oocito_ids - duck_oocito_ids
    
    print("\n=========================================================================")
    print("               DISCREPANCY SUMMARY: OOCITO_ID SET DIFFERENCE             ")
    print("=========================================================================")
    print(f"1. Oocito IDs present ONLY in DuckDB: {len(only_in_duck):,}")
    print(f"2. Oocito IDs present ONLY in Athena: {len(only_in_ath):,}")
    
    if only_in_duck:
        print("\n--- Sample Oocitos ONLY in DuckDB (Local) ---")
        sample_duck = list(only_in_duck)[:5]
        display(duck_con.execute(f"SELECT oocito_id, micro_prontuario, micro_data_procedimento, patient_unit_huntington FROM gold.embryos_with_prescription_wide WHERE oocito_id IN {tuple(sample_duck) if len(sample_duck) > 1 else f'({sample_duck[0]})'}").df())
        
    if only_in_ath:
        print("\n--- Sample Oocitos ONLY in AWS Athena (Prod) ---")
        sample_ath = list(only_in_ath)[:5]
        # Use embryo_unit_huntington in Athena query
        query_sample_ath = f"SELECT oocito_id, micro_prontuario, micro_data_procedimento, embryo_unit_huntington as patient_unit_huntington FROM {ATHENA_SCHEMA}.embryos_with_prescription_wide WHERE oocito_id IN {tuple(sample_ath) if len(sample_ath) > 1 else f'({sample_ath[0]})'}"
        try:
            ath_cur.execute(query_sample_ath)
            df_sample_ath = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
            display(df_sample_ath)
        except Exception as e:
            print(f"Error fetching sample oocitos from Athena: {e}")
else:
    print("\nAthena oocito IDs set is empty or query failed. Set difference skipped.")

## 🧼 Part 6: Value Discrepancy Diagnostics for Matching Oocitos

Reconciles a sample of medication doses for matching oocitos to ensure specific values are identical.

In [ ]:
if ath_cur is not None and len(ath_oocito_ids) > 0:
    common_oocitos = duck_oocito_ids.intersection(ath_oocito_ids)
    print(f"Common oocito count: {len(common_oocitos):,}")
    
    if len(common_oocitos) > 0:
        # Choose top 3 medication groups by count to verify doses
        top_groups = df_med_recon['Medication Group'].head(3).tolist()
        print(f"Verifying pivoted dose values for top 3 medication groups: {top_groups}")
        
        sample_size = min(1000, len(common_oocitos))
        sample_ids = list(common_oocitos)[:sample_size]
        
        # Fetch sample values from DuckDB
        cols_to_check_duck = ['oocito_id'] + [f"{g}_dose" for g in top_groups]
        query_duck_sample = f"SELECT {', '.join(cols_to_check_duck)} FROM gold.embryos_with_prescription_wide WHERE oocito_id IN {tuple(sample_ids) if len(sample_ids) > 1 else f'({sample_ids[0]})'}"
        df_duck_sample = duck_con.execute(query_duck_sample).df().set_index('oocito_id').sort_index()
        df_duck_sample.columns = [col.lower() for col in df_duck_sample.columns]
        
        # Fetch sample values from Athena
        cols_to_check_ath = ['oocito_id'] + [f"{g.lower()}_dose" for g in top_groups]
        query_ath_sample = f"SELECT {', '.join(cols_to_check_ath)} FROM {ATHENA_SCHEMA}.embryos_with_prescription_wide WHERE oocito_id IN {tuple(sample_ids) if len(sample_ids) > 1 else f'({sample_ids[0]})'}"
        try:
            ath_cur.execute(query_ath_sample)
            df_ath_sample = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description]).set_index('oocito_id').sort_index()
            df_ath_sample.columns = [col.lower() for col in df_ath_sample.columns]
            
            # Compare
            mismatches = {}
            for g in top_groups:
                col = f"{g.lower()}_dose"
                # Fill NaN for comparison
                duck_vals = df_duck_sample[col].fillna(-1.0).astype(float)
                ath_vals = df_ath_sample[col].fillna(-1.0).astype(float)
                
                # Align indexes to be safe
                duck_vals, ath_vals = duck_vals.align(ath_vals, join='inner')
                
                diffs = duck_vals != ath_vals
                mismatch_count = diffs.sum()
                if mismatch_count > 0:
                    print(f"Mismatch detected in column {col}: {mismatch_count} differences out of {len(duck_vals)}")
                    diff_idx = diffs[diffs].index[:5]
                    df_diff = pd.DataFrame({
                        'Local (DuckDB)': df_duck_sample.loc[diff_idx, col],
                        'Prod (Athena)': df_ath_sample.loc[diff_idx, col]
                    })
                    display(df_diff)
                else:
                    print(f"✅ All sampled records match perfectly in column: {col}")
        except Exception as e:
            print(f"Error during sample value comparison: {e}")
else:
    print("Skipped value reconciliation (Athena unavailable or no common oocitos).")

In [ ]:
# Close database connections
print("Closing database connections...")
try:
    duck_con.close()
    print("DuckDB connection closed.")
except Exception as e:
    print(f"Error closing DuckDB connection: {e}")
    
try:
    if ath_con:
        ath_con.close()
        print("Athena connection closed.")
except Exception as e:
    print(f"Error closing Athena connection: {e}")